<a href="https://colab.research.google.com/github/pksheaad/Transformers/blob/main/03_implementing_the_attention_block_PK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# Importing libraries
import torch
import torch.nn as nn
from torch.utils.data import RandomSampler
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader

import requests
from typing import Dict, List, Any

In [3]:
# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [4]:
# Retrieving the data
url = "https://raw.githubusercontent.com/pksheaad/Transformers/refs/heads/main/Data/tiny-shakespeare.txt"

request = requests.get(url = url)
text = request.text
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
class Char_Tokenizer:
  def __init__(self, vocabulary) -> None:
    self.token_id_to_char = {char:token_id for token_id, char in enumerate(vocabulary)}
    self.char_to_token_id = {token_id:char for token_id, char in enumerate(vocabulary)}

  @staticmethod
  def text_to_train(text:str):
    vocabulary = set(text)
    return Chat_Tokenizer(sorted(list(vocabulary)))

  def encode_text(self, text):
    token_ids = []
    for char in text:
      token_ids.append(self.token_id_to_char[char])

    return torch.tensor(token_ids, dtype = torch.long)

  def decode_text(self,token_ids):
    decoded_text = []
    for token in token_ids.tolist():
      decoded_text.append(self.char_to_token_id[token])

    return "".join(decoded_text)

  def get_length(self):
    return len(self.token_id_to_char)




In [7]:
# Testing
tokenizer = Char_Tokenizer.text_to_train(text = text)
print(tokenizer.encode_text(text)[:100])
print(tokenizer.decode_text(tokenizer.encode_text(text)[:100]))
print(tokenizer.get_length())

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
65


In [8]:
class TokenIdsDataset(Dataset):
  def __init__(self, data, block_size) -> None:
    super().__init__()
    self.data = data
    self.block_size = block_size

  def __len__(self):
    return len(self.data) - self.block_size

  def __getitem__(self, index):
    assert index < len(self.data) - self.block_size
    x = self.data[index : index + self.block_size]
    y = self.data[index + 1: index + 1 + self.block_size]

    return x, y


In [14]:
# Instantiate the dataset class
dataset = TokenIdsDataset(data = tokenizer.encode_text(text = text), block_size = 64)
dataset

In [15]:
# Create a Random Sampler
sampler = RandomSampler(data_source = dataset, replacement = True)
sampler

In [16]:
dataloader = DataLoader(dataset = dataset, batch_size = 64, sampler = sampler)
dataloader

In [23]:
x, y = next(iter(dataloader))
x, y
x.shape, y.shape

(torch.Size([64, 64]), torch.Size([64, 64]))

#Model Configuration
Before we start, we need to define a configuration for our model, containing hyperparameters that define its structure. This configuration is stored in a dictionary with the following parameters:

**Vocabulary Size**: The number of unique token IDs supported by the tokenizer.

**Context Size:** The maximum number of tokens the model can see at once.

**Embedding Dimension:** The size of the embedding vectors.

**Number of Heads:** The number of attention heads, each processing input independently.

**Number of Layers:** The number of Transformer blocks in the model.

**Dropout Rate:** The proportion of outputs set to zero in Dropout layers.

**Use Bias:** A boolean indicating whether the linear transformations should include bias terms.

In [22]:
config = {
    "vocabulary_size" : tokenizer.get_length(),
    "context_size" : 256,
    "embedding_dim" : 768,
    "num_head" : 12,
    "num_layer" : 10,
    "dropout" : 0.1,
    "use_bias" : False
}
config["head_size"] = config["embedding_dim"] // config["num_head"]

In [49]:
class AttentionHead(nn.Module):
  def __init__(self, config :Dict[str,Any]) -> None:
    super().__init__()
    # Q. K, V convert the input embedding vector to Q, K and V martices
    self.Q_weight = nn.Linear(in_features=config['embedding_dim'], out_features=config['head_size'], bias = config['use_bias'])
    self.K_weight = nn.Linear(in_features=config['embedding_dim'], out_features=config['head_size'], bias = config['use_bias'])
    self.V_weight = nn.Linear(in_features=config['embedding_dim'], out_features=config['head_size'], bias = config['use_bias'])

    # Dropout layer
    self.dropout = nn.Dropout(p = config['dropout'])

    # Masking Ensures the model cannot "cheat" by looking at future tokens.
    casual_mask_attention = torch.tril(torch.ones(config['context_size'], config['context_size']))
    self.register_buffer("casual_mask_attention", casual_mask_attention )

  def forward(self, input):
    #print(f"Input Shape: {input.shape}")
    batch_size, token_num, embedding_num = input.shape
    Q_Matrix = self.Q_weight(input)
    K_Matrix = self.K_weight(input)
    V_Matrix = self.V_weight(input)
    #print(f"Q Shape: {Q_Matrix.shape}|K Shape: {K_Matrix.shape}| V Shape: {V_Matrix.shape}")

    # get the attention score K X V
    attention_score = Q_Matrix @ K_Matrix.transpose(1,2)

    # Masking
    attention_score = attention_score.masked_fill(self.casual_mask_attention[:token_num,:token_num]==0,torch.inf)

    # Assurme tha attention score should not be bigger value
    attention_score = attention_score / ( K_Matrix.shape[-1] ** 0.5 )

    # Applying softmax
    attention_score = torch.softmax(attention_score, dim = -1)

    # Get the final embedding with attention score
    attention_score = attention_score @ V_Matrix

    #print(f"Shape of attention Score: {attention_score.shape}")

    return attention_score





In [44]:
# Testing AttentionHead Class
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input shape: {input.shape}")
ah = AttentionHead(config=config)
output = ah(input)
print(f"Output Shape: {output.shape}")

Input shape: torch.Size([8, 256, 768])
Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 64])


# Multi-Head Attention Implementatio

To implement the Multihead attention we implement

**Inherit from nn.Module**

**Create multiple heads** Instantiate Multiple AttentionHead instance and store them in Moulelist

**Dropout Layer** For regularization

In [45]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config:Dict[str, Any] ) -> None:
    super().__init__()

    # instantiate the mutiple AttentionHead
    head_list = [AttentionHead(config=config) for _ in range(config['num_head'])]
    # Store them into ModuleList
    self.heads = nn.ModuleList(head_list)
    # Linear layer
    self.linear = nn.Linear(in_features = config['head_size'] * config['num_head'], out_features=config['embedding_dim'])
    # Dropout layer
    self.dropout = nn.Dropout(p = config['dropout'])

  def forward(self, input):
    heads = [head(input) for head in self.heads]
    score_change = torch.cat(heads, dim = -1)
    score_chnage = self.linear(score_change)
    score_change = self.dropout(score_change)

    return score_change



In [50]:
mha = AttentionHead(config = config)
input = torch.rand(8, config['context_size'], config['embedding_dim'])
print(f"Input Shape: {input.shape}")
output = mha(input)
print(f"Output Shape: {output.shape}")

Input Shape: torch.Size([8, 256, 768])
Output Shape: torch.Size([8, 256, 64])
